In [1]:
# | default_exp gsc.sync

# GSC Sync
> Sync Google Search Console data by fetching from the API and storing in SQLite.

In [2]:
#| export
from sqlmodel import Session, select
from seootter.models import GSCAnalytics
from datetime import datetime, timedelta
from seootter.gsc.filters import store_gsc_data
from seootter.gsc_client import GSCAuth, fetch_gsc_data, get_date_range
import time

In [3]:
#| export
def store_single_date(session: Session,  # Active database session
                      auth: GSCAuth,  # Authenticated GSCAuth instance
                      site_url: str,  # GSC property URL
                      date: str  # Date to fetch and store (YYYY-MM-DD)
                      ) -> int:
    "Fetch and store GSC data for a single date. Returns number of detailed records stored."
    # Fetch exact property totals (without query dimension to avoid anonymization filtering)
    totals = fetch_gsc_data(auth, site_url, date, date, dimensions=["date"])
    store_gsc_property_totals(session, site_url, date, totals)
    
    # Fetch detailed query data
    rows = fetch_gsc_data(auth, site_url, date, date)
    store_gsc_data(session, site_url, date, rows)
    
    return len(rows)


In [4]:
#| export
def _sync_dates(session: Session,  # Active database session
                auth: GSCAuth,  # Authenticated GSCAuth instance
                site_url: str,  # GSC property URL
                dates: list[str]  # List of dates to sync (YYYY-MM-DD)
                ) -> dict:
    "Fetch and store GSC data for a list of dates, returning a results summary."
    results = {"successful": [], "failed": [], "total_records": 0}
    for date in dates:
        try:
            count = store_single_date(session, auth, site_url, date)
            results["successful"].append(date)
            results["total_records"] += count
            print(f"Synced {date}: {count} records")
        except Exception as e:
            results["failed"].append({"date": date, "error": str(e)})
            print(f"Failed {date}: {e}")
        time.sleep(1)
    return results

In [5]:
#| export
def store_date_range(session: Session, auth: GSCAuth, site_url: str, start_date: str, end_date: str) -> dict:
    "Fetch and store GSC data for all dates in range."
    return _sync_dates(session, auth, site_url, iter_dates(start_date, end_date))

In [6]:
#| export
def iter_dates(start_date: str,  # Start date (YYYY-MM-DD)
               end_date: str  # End date (YYYY-MM-DD)
               ) -> list[str]:
    "Generate all dates between start and end inclusive."
    start_d = datetime.strptime(start_date, "%Y-%m-%d").date()
    end_d = datetime.strptime(end_date, "%Y-%m-%d").date()
    return [(start_d + timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range((end_d - start_d).days + 1)]

In [7]:
#| export
def get_missing_dates(session: Session,  # Active database session
                      site_url: str,  # GSC property URL
                      start_date: str,  # Start date (YYYY-MM-DD)
                      end_date: str  # End date (YYYY-MM-DD)
                      ) -> list[str]:
    "Return dates in range that have no stored GSC data."
    stored = set(session.exec(
        select(GSCAnalytics.date).where(GSCAnalytics.site_url == site_url).distinct()
    ).all())
    return sorted(set(iter_dates(start_date, end_date)) - stored, reverse=True)

In [8]:
#| export
def sync_missing_dates(session: Session, auth: GSCAuth, site_url: str, start_date: str, end_date: str) -> dict:
    "Fetch and store GSC data for missing dates only."
    return _sync_dates(session, auth, site_url, get_missing_dates(session, site_url, start_date, end_date))

In [9]:
#| export
def daily_sync(session: Session,  # Active database session
               auth: GSCAuth,  # Authenticated GSCAuth instance
               sites: list[str]  # List of GSC property URLs to sync
               ) -> dict:
    "Sync missing GSC data for all sites up to today."
    _, end = get_date_range("today")
    return {site: sync_missing_dates(session, auth, site,
                                     min(get_missing_dates(session, site, "2020-01-01", end), default=end),
                                     end)
            for site in sites}